# City & Regional Yield Explorer

**Interactive** report built from the team's cleaned datasets
(`data/cleaned/cleaned_sale_properties.csv`, `cleaned_rent_properties.csv`).

* **Top chart** — number of properties *for sale* per `nearest_city`; bar **colour = gross
  rental ROI yield (%)** = `median rent €/m² × 12 ÷ median sale €/m²`, computed against comparable
  rent listings in the same city.
* **Pick a city from the dropdown** → the bottom chart shows that city's yield **broken down by
  distance band** (0–1, 1–2.5, 2.5–5, 5–10, 10–15, 15–20, >20 km). Each band again shows the
  number of properties for sale, coloured by the yield against comparable rent listings.

> Pure Plotly — renders in VS Code, Jupyter and exported HTML with no widget runtime needed.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

pd.set_option("display.max_columns", None)

In [2]:
# --- locate the cleaned datasets (works whether cwd is repo root or reports/) ---
DATA_DIR = next(
    p for p in [Path("../data/cleaned"), Path("data/cleaned"), Path("./data/cleaned")]
    if (p / "cleaned_sale_properties.csv").exists()
)
SALE_CSV = DATA_DIR / "cleaned_sale_properties.csv"
RENT_CSV = DATA_DIR / "cleaned_rent_properties.csv"

# market-bound guard rails (mirror src/yieldinsight_v2.py) so yields stay sane
SALE_PRICE_MIN, SALE_PRICE_MAX = 25_000, 15_000_000
RENT_PRICE_MIN, RENT_PRICE_MAX = 200, 25_000
SURFACE_MIN,    SURFACE_MAX    = 9, 3_000
SALE_PPSQM_MIN, SALE_PPSQM_MAX = 400, 18_000
RENT_PPSQM_MIN, RENT_PPSQM_MAX = 3, 70


def load_clean(path: Path, kind: str) -> pd.DataFrame:
    """Load a listings CSV, apply guard rails, recompute the authoritative €/m²."""
    df = pd.read_csv(path, low_memory=False)
    for c in ["price", "livable_surface", "nearest_city_distance_km"]:
        df[c] = pd.to_numeric(df[c], errors="coerce")
    df = df.dropna(subset=["price", "livable_surface", "nearest_city"])
    df = df[df["livable_surface"].between(SURFACE_MIN, SURFACE_MAX)]
    df["ppsqm"] = df["price"] / df["livable_surface"]
    if kind == "sale":
        df = df[df["price"].between(SALE_PRICE_MIN, SALE_PRICE_MAX)]
        df = df[df["ppsqm"].between(SALE_PPSQM_MIN, SALE_PPSQM_MAX)]
    else:
        df = df[df["price"].between(RENT_PRICE_MIN, RENT_PRICE_MAX)]
        df = df[df["ppsqm"].between(RENT_PPSQM_MIN, RENT_PPSQM_MAX)]
    return df.reset_index(drop=True)


sale = load_clean(SALE_CSV, "sale")
rent = load_clean(RENT_CSV, "rent")
print(f"clean sale = {len(sale):,}   clean rent = {len(rent):,}   cities = {sale['nearest_city'].nunique()}")

clean sale = 12,927   clean rent = 4,358   cities = 30


In [3]:
# --- distance bands + yield helper ----------------------------------------
DIST_BINS   = [0, 1, 2.5, 5, 10, 15, 20, np.inf]
DIST_LABELS = ["0–1 km", "1–2.5 km", "2.5–5 km", "5–10 km",
               "10–15 km", "15–20 km", ">20 km"]

MIN_RENT_N = 5   # min comparable rent listings before we trust a yield


def yield_pct(rent_ppsqm, sale_ppsqm):
    """Gross annual rental yield % = rent €/m² × 12 ÷ sale €/m² × 100."""
    if not sale_ppsqm or np.isnan(sale_ppsqm) or rent_ppsqm is None or np.isnan(rent_ppsqm):
        return np.nan
    return rent_ppsqm * 12 / sale_ppsqm * 100

In [4]:
# --- per-city table: count for sale + gross yield vs comparable rent -------
def city_table(sale: pd.DataFrame, rent: pd.DataFrame) -> pd.DataFrame:
    sg = sale.groupby("nearest_city").agg(n_sale=("ppsqm", "size"),
                                          sale_ppsqm=("ppsqm", "median"))
    rg = rent.groupby("nearest_city").agg(n_rent=("ppsqm", "size"),
                                          rent_ppsqm=("ppsqm", "median"))
    t = sg.join(rg, how="left")
    t["rent_ppsqm"] = t["rent_ppsqm"].where(t["n_rent"] >= MIN_RENT_N)  # drop thin rent samples
    t["yield"] = t["rent_ppsqm"] * 12 / t["sale_ppsqm"] * 100
    return t.sort_values("n_sale", ascending=False).reset_index()


cities = city_table(sale, rent)
cities.round(1).head(10)

,nearest_city,n_sale,sale_ppsqm,n_rent,rent_ppsqm,yield
0,Brussels,3449,4067.0,2226.0,16.1,4.8
1,Charleroi,860,1745.5,147.0,9.9,6.8
2,Namur,854,2893.3,252.0,10.3,4.3
3,Antwerp,645,2899.2,268.0,14.0,5.8
4,Liège,629,2432.4,134.0,10.1,5.0
5,Seraing,560,2248.6,124.0,9.5,5.1
6,Ghent,494,3529.9,77.0,10.2,3.5
7,Mechelen,463,2940.6,88.0,10.6,4.3
8,Tournai,456,2519.6,155.0,9.2,4.4
9,Aalst,453,2613.6,49.0,10.0,4.6


In [5]:
# --- per-city distance breakdown ------------------------------------------
def city_detail(city: str) -> pd.DataFrame:
    """Yield per distance band for one city; rent comparable falls back to
    the city-wide median when a band has too few rent listings."""
    s = sale[sale["nearest_city"] == city].copy()
    r = rent[rent["nearest_city"] == city].copy()
    s["band"] = pd.cut(s["nearest_city_distance_km"], DIST_BINS, labels=DIST_LABELS, right=False)
    r["band"] = pd.cut(r["nearest_city_distance_km"], DIST_BINS, labels=DIST_LABELS, right=False)

    city_rent_ppsqm = r["ppsqm"].median() if len(r) >= MIN_RENT_N else np.nan
    rows = []
    for lab in DIST_LABELS:
        sb = s[s["band"] == lab]
        rb = r[r["band"] == lab]
        n_sale = len(sb)
        sale_ppsqm = sb["ppsqm"].median() if n_sale else np.nan
        if len(rb) >= MIN_RENT_N:                    # enough comparables in this band
            rent_ppsqm, basis = rb["ppsqm"].median(), "band rent"
        else:                                        # fall back to city-wide rent
            rent_ppsqm, basis = city_rent_ppsqm, "city rent"
        rows.append(dict(band=lab, n_sale=n_sale, n_rent=len(rb),
                         sale_ppsqm=sale_ppsqm, rent_ppsqm=rent_ppsqm,
                         yield_pct=yield_pct(rent_ppsqm, sale_ppsqm), basis=basis))
    return pd.DataFrame(rows)


city_detail(cities["nearest_city"].iloc[0]).round(1)

,band,n_sale,n_rent,sale_ppsqm,rent_ppsqm,yield_pct,basis
0,0–1 km,131,229,4567.6,20.0,5.3,band rent
1,1–2.5 km,334,297,3903.0,18.0,5.5,band rent
2,2.5–5 km,1073,740,3984.0,16.2,4.9,band rent
3,5–10 km,1300,747,4529.9,15.9,4.2,band rent
4,10–15 km,231,87,3495.1,12.4,4.2,band rent
5,15–20 km,229,106,3724.6,12.2,3.9,band rent
6,>20 km,151,20,3041.7,9.9,3.9,band rent


In [6]:
# --- interactive figure: pick a city from the dropdown -> distance breakdown
CSCALE = "RdYlGn"   # red = low yield, green = high yield
_valid = cities["yield"].dropna()
CMIN, CMAX = float(np.floor(_valid.quantile(0.05))), float(np.ceil(_valid.quantile(0.95)))

city_list = cities["nearest_city"].tolist()
details = {c: city_detail(c) for c in city_list}    # precompute every breakdown

fig = make_subplots(
    rows=2, cols=1, vertical_spacing=0.22, row_heights=[0.5, 0.5],
    subplot_titles=("Properties for sale per nearest city — colour = gross rental yield (%)",
                    "Distance breakdown for the selected city"))

# row 1 — overview, always visible (trace 0); only this trace carries the colorbar
fig.add_bar(
    row=1, col=1, x=cities["nearest_city"], y=cities["n_sale"], showlegend=False,
    marker=dict(color=cities["yield"], colorscale=CSCALE, cmin=CMIN, cmax=CMAX,
                colorbar=dict(title="Gross<br>yield %")),
    customdata=np.stack([cities["yield"], cities["sale_ppsqm"],
                         cities["rent_ppsqm"], cities["n_rent"].fillna(0)], axis=-1),
    hovertemplate="<b>%{x}</b><br>%{y:,} for sale<br>yield %{customdata[0]:.1f}%<br>"
                  "buy €%{customdata[1]:,.0f}/m² · rent €%{customdata[2]:.1f}/m² "
                  "(n=%{customdata[3]:.0f})<extra></extra>")

# row 2 — one detail trace per city (traces 1..N); only the first is visible
for i, c in enumerate(city_list):
    d = details[c]
    cd = d[["yield_pct", "sale_ppsqm", "rent_ppsqm", "n_rent", "basis"]].to_numpy()  # object dtype keeps str
    fig.add_bar(
        row=2, col=1, x=d["band"], y=d["n_sale"], name=c, visible=(i == 0), showlegend=False,
        marker=dict(color=d["yield_pct"], colorscale=CSCALE, cmin=CMIN, cmax=CMAX, showscale=False),
        customdata=cd,
        hovertemplate="<b>%{x}</b><br>%{y:,} for sale<br>yield %{customdata[0]:.1f}%<br>"
                      "buy €%{customdata[1]:,.0f}/m² · rent €%{customdata[2]:.1f}/m²<br>"
                      "basis: %{customdata[4]} (n=%{customdata[3]:.0f})<extra></extra>")

# dropdown — each button shows the overview + exactly one city's detail trace
n = len(city_list)
buttons = [dict(label=c, method="update",
                args=[{"visible": [True] + [j == i for j in range(n)]}])
           for i, c in enumerate(city_list)]

fig.update_layout(
    template="plotly_white", height=830, bargap=0.25,
    margin=dict(t=95, l=70, r=20, b=120),
    updatemenus=[dict(buttons=buttons, active=0, direction="down", showactive=True,
                      x=1.0, xanchor="right", y=1.075, yanchor="bottom",
                      pad=dict(t=4, b=4), bgcolor="white", bordercolor="#bbbbbb")],
    annotations=list(fig.layout.annotations) + [
        dict(text="Pick a city ▸", x=0.78, y=1.085, xref="paper", yref="paper",
             showarrow=False, font=dict(size=12, color="#555555"))])

fig.update_xaxes(title_text="nearest city", tickangle=-40, row=1, col=1)
fig.update_yaxes(title_text="# properties for sale", row=1, col=1)
fig.update_xaxes(title_text="distance from nearest city", row=2, col=1)
fig.update_yaxes(title_text="# properties for sale", row=2, col=1)
fig